# 4.1 怎麼知道字的順序？

# 第 4 章：第一個 Dense Transformer 語言模型

前置：第3章的因果 attention。完整可讀實作在 tiny_perceptron/model.py；預設只用 learned position、LayerNorm、Dense FFN，還沒有 FlashAttention 或 MoE。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：像原稿旁加修訂，而不是撕掉原稿**

y=x+f(x)，原資料和修訂都有路徑。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/residual.svg")))

**先想一想：**相同字在第一格與第三格，向量還一樣嗎？

沒有位置資訊時，交換前文順序可能只留下相同內容集合。把每個位置自己的向量加進輸入，讓「第幾格」成為可辨線索。

**把直覺寫成數學：**$x_t=E[token_t]+P[t]$，兩者 feature 維度必須相同。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
e, p = nn.Embedding(4, 3), nn.Embedding(5, 3)
ids = torch.tensor([1, 2, 1])
x = e(ids) + p(torch.arange(3))
print(x)
assert not torch.equal(x[0], x[2])

**如何讀結果：**這是 learned absolute positions；RoPE 留到14.1，不能超過position表長度。

**只改一件事：**只把位置向量全設為0，再比較相同字。
